<a href="https://colab.research.google.com/github/santimontouliu/MyPythonPortfolio/blob/master/Franck_Hertz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

def simulate_franck_hertz(gas='Mercury', temp_c=180, v_contact=1.5, noise_level=0.02):
    # Constants for gases
    gas_data = {
        'Mercury': {'E_exc': 4.9, 'I_scale': 1.0, 'color': 'blue'},
        'Neon': {'E_exc': 18.7, 'I_scale': 0.5, 'color': 'orange'}
    }

    E_exc = gas_data[gas]['E_exc']
    V_accel = np.linspace(0, 40, 400)

    # 1. Base current following Space-Charge Limit (Child-Langmuir Law: I ~ V^1.5)
    # We shift V_accel by the contact potential
    V_eff = np.maximum(0, V_accel - v_contact)
    I_base = 0.1 * (V_eff**1.5)

    # 2. Modeling the dips using a periodic attenuation function
    # The dips occur every E_exc. We use a sine-based mod to create periodic losses.
    # The width of the dip is affected by thermal spread (Temperature)
    thermal_spread = 0.1 + (temp_c / 500)
    dips = 0.5 * (1 + np.cos(2 * np.pi * V_eff / E_exc))

    # Inelastic collision efficiency (Current drops near multiples of E_exc)
    attenuation = 1 - (0.6 * np.exp(-((V_eff % E_exc) - E_exc)**2 / (2 * thermal_spread**2)))
    attenuation *= 1 - (0.6 * np.exp(-((V_eff % E_exc) - 0)**2 / (2 * thermal_spread**2)))

    # 3. Combine and add random electronic noise
    I_measured = I_base * attenuation * gas_data[gas]['I_scale']
    noise = np.random.normal(0, noise_level, size=V_accel.shape)
    I_measured = np.maximum(0, I_measured + noise)

    # Plotting
    plt.figure(figsize=(10, 6))
    plt.plot(V_accel, I_measured, label=f'Measured Current ({gas})', color=gas_data[gas]['color'], lw=2)
    plt.title(f"Franck-Hertz Experiment Simulation: {gas}")
    plt.xlabel("Accelerating Voltage $V_{G2}$ (Volts)")
    plt.ylabel("Anode Current $I_A$ (Arbitrary Units)")
    plt.grid(True, linestyle='--', alpha=0.7)

    # Annotate peaks for the first two
    plt.annotate('1st Peak', xy=(E_exc + v_contact, I_measured[int((E_exc+v_contact)*10)]),
                 xytext=(E_exc + v_contact + 2, 5), arrowprops=dict(arrowstyle='->'))

    plt.legend()
    plt.show()

# Create Interactive UI
interact(simulate_franck_hertz,
         gas=Dropdown(options=['Mercury', 'Neon'], value='Mercury', description='Gas Type:'),
         temp_c=FloatSlider(min=140, max=220, step=5, value=180, description='Temp (°C):'),
         v_contact=FloatSlider(min=0.5, max=2.5, step=0.1, value=1.5, description='V_contact:'),
         noise_level=FloatSlider(min=0.0, max=0.1, step=0.01, value=0.02, description='Inst. Noise:'))

interactive(children=(Dropdown(description='Gas Type:', options=('Mercury', 'Neon'), value='Mercury'), FloatSl…

<function __main__.simulate_franck_hertz(gas='Mercury', temp_c=180, v_contact=1.5, noise_level=0.02)>